# Native NeMo vs sherpa-onnx Parakeet TDT Quality Comparison

This notebook runs native NeMo inference on the same FLEURS Italian samples
used in the sherpa-onnx benchmark to isolate the pipeline vs model quality gap.

**Purpose**: Determine whether the 3.0% (NVIDIA) vs 4.3% (sherpa-onnx) WER gap
on Italian FLEURS comes from:
- INT8 quantization quality loss
- sherpa-onnx feature extraction differences
- Different text normalization

**Hardware**: GPU required (T4 or better).

In [ ]:
# Install NeMo
!pip install -U nemo_toolkit['asr'] soundfile

In [ ]:
import json
import re
import numpy as np
import nemo.collections.asr as nemo_asr

def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[.,;:!?\"()\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def wer(reference: str, hypothesis: str) -> float:
    ref_words = normalize(reference).split()
    hyp_words = normalize(hypothesis).split()
    if not ref_words:
        return 0.0
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[n][m] / n

In [ ]:
# Load the Parakeet TDT model from HuggingFace (FP32/FP16, no quantization)
model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained('nvidia/parakeet-tdt-0.6b-v3')
print('Model loaded successfully')

In [ ]:
# Upload FLEURS samples from your local /tmp/parakeet-bench/samples/ directory
# and the references.json file.

from google.colab import files
import os

os.makedirs('/tmp/nemo-bench/samples', exist_ok=True)

print('Upload references.json and all it_fleurs_*.wav files:')
uploaded = files.upload()

for filename in uploaded:
    if filename.endswith('.wav'):
        os.rename(filename, f'/tmp/nemo-bench/samples/{filename}')
    elif filename == 'references.json':
        os.rename(filename, '/tmp/nemo-bench/references.json')

In [ ]:
# Run NeMo transcription on each sample
with open('/tmp/nemo-bench/references.json') as f:
    samples = json.load(f)

print(f'Loaded {len(samples)} samples')

results = []
for i, sample in enumerate(samples):
    hypotheses = model.transcribe([sample['path']], batch_size=1)
    hypothesis = hypotheses[0].text if hasattr(hypotheses[0], 'text') else str(hypotheses[0])
    reference = sample['reference'].strip()
    sample_wer = wer(reference, hypothesis)
    results.append({
        'index': i,
        'name': sample.get('name', f'sample_{i}'),
        'wer': sample_wer,
        'reference': reference,
        'hypothesis': hypothesis,
    })
    print(f'[{i}] {sample_wer:>5.1%} WER  {sample.get("name", "")}')
    print(f'     REF: {reference[:80]}')
    print(f'     HYP: {hypothesis[:80]}')

avg_wer = sum(r['wer'] for r in results) / len(results)
print(f'\nNative NeMo average WER: {avg_wer:.1%}')

In [ ]:
# Upload sherpa-onnx results for comparison
print('Upload mel_config_ab_results.json:')
uploaded = files.upload()
sherpa_results_path = list(uploaded.keys())[0]

with open(sherpa_results_path) as f:
    sherpa_data = json.load(f)

print('=' * 70)
print('  NATIVE NeMo (FP32) vs sherpa-onnx (INT8) COMPARISON')
print('=' * 70)
print(f'  {"Sample":<20} {"NeMo FP32":>12} {"sherpa INT8":>12} {"Delta":>8}')
print(f'  {"─" * 55}')

for nemo_r in results:
    sherpa_sample = next(
        (s for s in sherpa_data['samples'] if s['name'] == nemo_r['name']),
        None
    )
    if sherpa_sample:
        sherpa_wer = sherpa_sample['results_per_config'][0]['wer']
        delta = nemo_r['wer'] - sherpa_wer
        print(f'  {nemo_r["name"]:<20} {nemo_r["wer"]:>11.1%} {sherpa_wer:>11.1%} {delta:>+7.1%}')

nemo_avg = sum(r['wer'] for r in results) / len(results)
sherpa_fleurs = [s for s in sherpa_data['samples'] if s['name'].startswith('fleurs')]
sherpa_avg = sum(s['results_per_config'][0]['wer'] for s in sherpa_fleurs) / len(sherpa_fleurs)

print(f'  {"─" * 55}')
print(f'  {"AVERAGE":<20} {nemo_avg:>11.1%} {sherpa_avg:>11.1%} {nemo_avg - sherpa_avg:>+7.1%}')
print(f'\n  INT8 quantization quality loss: {sherpa_avg - nemo_avg:+.1%}')